# Constrained sampling on real data: anchored Langevin with a **state-dependent** $J$

This is `Constrained_Sampling_ANDS_ball_constrained.ipynb` carried over from the synthetic
3-dimensional target to two real binary-classification posteriors — **Titanic** and the
**MAGIC Gamma Telescope** — with the same figure set: $W_1$ per coordinate, 2-d contours with
the constraint drawn, a sweep over the skew strength, and a fixed-physical-time stepsize
control.

Three things change on real data, and the third is the one worth the notebook.

1. **The target is a Gibbs posterior, non-differentiable twice over.** For labels
   $y_i\in\{-1,+1\}$ and standardised features $\phi_i$, write $\psi_i=y_i\phi_i$ and

   $$U(w)\;=\;\tau\,\frac1n\sum_{i=1}^n\max\!\bigl(0,\,1-w^{\!\top}\psi_i\bigr)\;+\;\lambda\lVert w\rVert_1,
   \qquad \pi(w)\;\propto\;e^{-U(w)}\,\mathbf 1_K(w),$$

   on the ball $K=\{\lVert w\rVert_2\le R\}$. The hinge kinks at every active margin and the
   $\ell_1$ prior kinks on every coordinate axis. $U$ is only ever **evaluated**, never
   differentiated — which is the whole premise of the anchored scheme.

2. **There is no analytic reference.** The synthetic notebook could draw exact samples by
   rejection. Here the reference is a long random-walk Metropolis chain run against the
   *exact* non-smooth $U$ with proposals rejected outside $K$, which is reversible for
   $\pi\mathbf 1_K$ with no projection bias. A second, independent reference chain sets the
   floor: the $W_1$ below which two samples of this size are indistinguishable.

3. **$d$ is 10 and 11, not 3 — and that turns the correction term on.** In the original
   notebook the axial field happened to be divergence-free, so the correction could be
   dropped. In $d>3$ it cannot. That is Experiment 3.

## What the anchored dynamics needs from $J$

For constant skew $J$ the extra stationary flux is $J\nabla\varphi$ with $\varphi=e^{-U_0}$, and
$\nabla\!\cdot(J\nabla\varphi)=0$ follows from antisymmetry alone. Once $J$ varies with $x$ that
argument fails; the correct divergence-free flux is $\mathcal F_i=\sum_j\partial_j(J_{ij}\varphi)$,
whose divergence vanishes because $J_{ij}\varphi$ is antisymmetric in $(i,j)$ while
$\partial_i\partial_j$ is symmetric. Dividing by $\pi$ gives the **anchored, state-dependent-$J$
dynamics**

$$\boxed{\;dX_t=e^{\Delta(X_t)}\Bigl[-\bigl(I+J(X_t)\bigr)\nabla U_0(X_t)+\nabla\!\cdot J(X_t)\Bigr]dt
\;+\;\sqrt2\,e^{\Delta(X_t)/2}\,dW_t\;}$$

with $(\nabla\!\cdot J)_i=\sum_j\partial_j J_{ij}$ and $\Delta=U-U_0$.

Sampling on $K$ by projection adds a **second, independent** condition: the skew drift must be
tangential to the wall,

$$J(x)\,\nu(x)=0\quad\text{for }x\in\partial K,\qquad\nu(x)=x/\lVert x\rVert,$$

or the skew term pushes probability into the boundary, the projection absorbs it, and the
stationary law is distorted by an amount no stepsize refinement removes.

### The three fields, generalised to $d$ dimensions

The original notebook's axial field $J_s(x)w=s\,(x\times w)$ is a cross product and exists only
in $d=3$. Its $d$-dimensional replacement, for a fixed antisymmetric $A$ (the tridiagonal skew,
normalised to unit spectral norm), is

$$J_s(x)\;=\;\frac{s}{R^2}\Bigl[\lVert x\rVert^2A\;+\;x\,(Ax)^{\!\top}-\,(Ax)\,x^{\!\top}\Bigr].$$

It is antisymmetric for every $x$; it annihilates $x$ **identically**, $J_s(x)x=0$, so
$J_s\nu=0$ on $\partial K$ by construction; it is polynomial, so there is no singularity at the
origin; and the $1/R^2$ makes $\lVert J_s\rVert$ on the wall comparable to $\lVert J_a\rVert$ at
the same $s$. Its divergence is available in closed form:

$$(\nabla\!\cdot J_s)(x)\;=\;-\frac{s}{R^2}\,(d-2)\,Ax .$$

| | $J(x)$ | $\nabla\!\cdot J$ | $J(x)\nu(x)$ on $\partial K$ |
|---|---|---|---|
| `none` | $0$ | $0$ | $0$ |
| `const` | $s\,A$ (constant) | $0$ | $s\,A\nu\neq0$ — **violates the boundary condition** |
| `axial` | $\tfrac{s}{R^2}\bigl[r^2A+x(Ax)^{\!\top}-(Ax)x^{\!\top}\bigr]$ | $-\tfrac{s}{R^2}(d-2)Ax\neq0$ — **correction is mandatory** | $0$ — satisfied identically |

Note the $(d-2)$: the correction vanishes at $d=2$, and at $d=3$ it is small, which is why the
original notebook could get away with a divergence-free 3-d field. At $d=10$ and $d=11$ it is the
same order as the drift itself. Both claims are checked numerically below rather than taken on
faith.

In [ ]:
%matplotlib inline
import os, sys, time, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import torch
torch.set_default_dtype(torch.float64)
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", context="notebook")

from ands import data as D, diagnostics as G, experiments as E, plots as P, skew as SK

CFG = E.CONFIG
print(json.dumps({k: v for k, v in CFG.items()}, indent=2))

## The two datasets

In [ ]:
DS = {"titanic": D.load_titanic(), "magic": D.load_magic(2000)}
rows = []
for key, ds in DS.items():
    rows.append({"dataset": key, "title": ds.title, "n": ds.n, "d": ds.d,
                 "P(y=+1)": round(float((ds.y > 0).mean()), 3),
                 "R": E.DATASETS[key]["R"],
                 "sha256[:16]": D.file_digest(key)})
display(pd.DataFrame(rows).set_index("dataset"))
for key, ds in DS.items():
    print("{:8s} {}".format(key, ds.feature_names))

### The anchor

The hinge term is an **average**, not a sum. That matters: the smoothing gap of the anchor is
then $O(\tau\delta)$ regardless of the sample size $n$, so $e^{\Delta}=e^{U-U_0}$ stays a healthy
$O(1)$ time change. With a sum, the gap would grow like $n\tau\delta$ and $e^{\Delta}$ would
underflow to zero on any real dataset — the anchored scheme would silently stop moving.

Both smoothings dominate what they replace ($\sqrt{z^2+\delta^2}\ge|z|$), so $U_0\ge U$ and
$e^{\Delta}\in(0,1]$.

In [ ]:
rows = []
for key in DS:
    tgt = E.build_target(key)
    g = torch.Generator().manual_seed(0)
    w = tgt.project(torch.randn(4000, tgt.d, generator=g) * (tgt.R / np.sqrt(tgt.d)) * 2)
    Delta = tgt.anchor_gap(w)
    bound = tgt.tau * tgt.delta / 2 + tgt.lam * tgt.d * tgt.delta
    rows.append({"dataset": key,
                 "U0 >= U everywhere": bool((tgt.U0(w) >= tgt.U(w) - 1e-12).all()),
                 "min Delta": round(float(Delta.min()), 4),
                 "worst-case bound": round(-bound, 4),
                 "min e^Delta": round(float(torch.exp(Delta).min()), 4),
                 "mean |grad U0|": round(float(tgt.grad_U0(w).norm(dim=1).mean()), 2)})
display(pd.DataFrame(rows).set_index("dataset"))

## Check 1 — the two conditions, numerically

`autograd_divergence` differentiates $J_{ij}(x)$ entry by entry and contracts, with no knowledge
of the closed forms. Agreement to $10^{-15}$ is the statement that
$\nabla\!\cdot J_s=-\frac{s}{R^2}(d-2)Ax$ is right.

In [ ]:
S_MAIN = CFG["s_main"]
rows = []
for key in DS:
    tgt, d, R = E.build_target(key), DS[key].d, E.DATASETS[key]["R"]
    g = torch.Generator().manual_seed(11)
    x = torch.randn(300, d, generator=g)
    x_wall = R * x / x.norm(dim=1, keepdim=True)          # points on dK
    for kind in ("const", "axial"):
        f = SK.build_skew(kind, d, S_MAIN, radius=R)
        div_auto = SK.autograd_divergence(f, x[:8])
        rows.append({
            "dataset": key, "field": kind, "d": d,
            "max |J + J^T|": "{:.1e}".format(SK.antisymmetry_error(f, x[:8])),
            "|div_autograd - div_closed|": "{:.1e}".format(
                float((div_auto - f.divergence(x[:8])).abs().max())),
            "mean ||div J|| on dK": round(float(f.divergence(x_wall).norm(dim=1).mean()), 3),
            "mean ||J(x) nu|| on dK": "{:.2e}".format(
                float(SK.boundary_flux(f, x_wall / R).mean())),
            "mean ||J||_2 on dK": round(SK.mean_operator_norm(f, x_wall), 3)})
display(pd.DataFrame(rows).set_index(["dataset", "field"]))

## Check 2 — the exact reference

Two independent random-walk Metropolis runs against the exact $U$, proposals rejected outside
$K$. The acceptance rate is tuned to the Roberts–Gelman–Gilks optimum by a pilot covariance. The
`W1 floor` is the mean $W_1$ between equally sized draws from the two runs: a method at the floor
is statistically indistinguishable from the truth at this sample size, and nothing below it means
anything.

In [ ]:
REF = {}
for key in DS:
    REF[key] = E.reference(key)
    r = REF[key]
    print("         classification accuracy at the posterior mean: {:.3f}".format(
        r["target"].accuracy(r["ref"])))

## Experiment 1 — $J=0$ vs constant $J_a$ vs state-dependent $J_s$

Same walkers, same seed, same stepsize; the only difference is the skew field. The state-dependent
run carries its $\nabla\!\cdot J$ correction.

In [ ]:
METHODS = [("none", 0.0, False), ("const", S_MAIN, False), ("axial", S_MAIN, False)]
EXP1 = {}
for key in DS:
    tgt, ref = REF[key]["target"], REF[key]["ref"]
    EXP1[key] = {}
    for kind, s, drop in METHODS:
        lab = E.METHOD_LABEL[kind]
        t0 = time.time()
        EXP1[key][lab] = E.chain(key, kind, s, ref=ref, drop_correction=drop)
        print("{:8s} {:28s} {:5.0f}s".format(key, lab.replace("$", ""), time.time() - t0))

    tab = []
    for lab, r in EXP1[key].items():
        sc = E.score(ref, r["x"], tgt.R)
        tab.append({"method": lab.replace("$", ""),
                    "mean W1": round(sc["W1_mean"], 4),
                    "worst-coordinate W1": round(float(sc["W1"].max()), 4),
                    "max KS": round(sc["maxKS"], 4),
                    "mass near dK": "{:.2%}".format(sc["boundary"])})
    tab.append({"method": "reference (truth)", "mean W1": round(REF[key]["floor"].mean(), 4),
                "worst-coordinate W1": round(float(REF[key]["floor"].max()), 4), "max KS": 0.0,
                "mass near dK": "{:.2%}".format(G.boundary_mass(ref, tgt.R))})
    print("\n== {} ==".format(DS[key].title))
    display(pd.DataFrame(tab).set_index("method"))

In [ ]:
for key in DS:
    worst = int(np.argmax(E.score(REF[key]["ref"], EXP1[key][E.METHOD_LABEL["const"]]["x"],
                                  REF[key]["target"].R)["W1"]))
    coords = sorted({1, 2, worst})[:3]
    P.w1_traces(EXP1[key], REF[key]["floor"], DS[key].feature_names,
                DS[key].title + r"  —  $W_1$ to the exact reference",
                "w1_{}".format(key), coords=coords)
    plt.show()

In [ ]:
for key in DS:
    P.density_panels(REF[key]["ref"], EXP1[key], E.DATASETS[key]["R"],
                     DS[key].title + "  —  2-d marginals with the constraint drawn",
                     "density_{}".format(key), dims=(1, 2),
                     feature_names=DS[key].feature_names)
    plt.show()

## Experiment 2 — sweep over the skew strength

If the boundary condition is what matters, the penalty should grow with how hard the inadmissible
field pushes, and the admissible one should be flat.

In [ ]:
SWEEP = {}
for key in DS:
    tgt, ref = REF[key]["target"], REF[key]["ref"]
    SWEEP[key] = {}
    for kind in ("const", "axial"):
        lab = E.METHOD_LABEL[kind]
        ss, w1, bd, ks = [], [], [], []
        for s in CFG["sweep"]:
            r = E.chain(key, "none" if s == 0 else kind, s, ref=None, track=False)
            sc = E.score(ref, r["x"], tgt.R)
            ss.append(s); w1.append(sc["W1_mean"]); bd.append(sc["boundary"]); ks.append(sc["maxKS"])
        SWEEP[key][lab] = {"s": ss, "W1_mean": w1, "boundary": bd, "maxKS": ks}

    tab = {}
    for lab, r in SWEEP[key].items():
        tab[lab.replace("$", "") + " : mean W1"] = [round(v, 4) for v in r["W1_mean"]]
        tab[lab.replace("$", "") + " : mass near dK"] = ["{:.2%}".format(v) for v in r["boundary"]]
    print("\n== {} ==".format(DS[key].title))
    display(pd.DataFrame(tab, index=["s = {:g}".format(s) for s in CFG["sweep"]]).T)

In [ ]:
for key in DS:
    P.sweep_panels(SWEEP[key], REF[key]["floor"].mean(),
                   G.boundary_mass(REF[key]["ref"], E.DATASETS[key]["R"]),
                   DS[key].title + "  —  cost of violating $J\\nu = 0$",
                   "sweep_{}".format(key))
    plt.show()

## Experiment 3 — the correction term is not optional

This is the experiment the 3-dimensional notebook could not run. There $\nabla\!\cdot J_s$ was
exactly zero, so the state-dependent update coincided with the constant-$J$ one and the theory's
correction term was a formality. At $d=10$ and $d=11$ it is not: dropping
$+\,e^{\Delta}\nabla\!\cdot J$ leaves a field that is antisymmetric and tangential — it passes both
checks above — and still samples the wrong distribution, because the flux it generates is no longer
divergence-free.

In [ ]:
ABLATION = {}
for key in DS:
    tgt, ref = REF[key]["target"], REF[key]["ref"]
    ABLATION[key] = {}
    for drop, kind in [(False, "axial"), (True, "axial_nocorr")]:
        lab = E.METHOD_LABEL[kind]
        ss, w1, bd = [], [], []
        for s in CFG["sweep"]:
            r = E.chain(key, "axial", s, ref=None, track=False, drop_correction=drop)
            sc = E.score(ref, r["x"], tgt.R)
            ss.append(s); w1.append(sc["W1_mean"]); bd.append(sc["boundary"])
        ABLATION[key][lab] = {"s": ss, "W1_mean": w1, "boundary": bd}

    tab = {lab.replace("$", ""): [round(v, 4) for v in r["W1_mean"]]
           for lab, r in ABLATION[key].items()}
    tab["reference floor"] = [round(REF[key]["floor"].mean(), 4)] * len(CFG["sweep"])
    print("\n== {} ==  mean W1".format(DS[key].title))
    display(pd.DataFrame(tab, index=["s = {:g}".format(s) for s in CFG["sweep"]]).T)

In [ ]:
for key in DS:
    P.ablation_panel(ABLATION[key], REF[key]["floor"].mean(),
                     DS[key].title + r"  —  keeping vs dropping $\nabla\cdot J$",
                     "ablation_{}".format(key))
    plt.show()

## Experiment 4 — which errors vanish with the stepsize, and which do not

Two different errors live on the boundary and the same figure cannot separate them. Projection
deposits an atom on $\partial K$ that the continuous target does not have; that is a discretisation
artefact and should shrink like $\sqrt\eta$. A violated boundary condition is a bias in the
stationary law and should **not** shrink at all.

To see them apart, the horizon is held at a fixed physical time $T=\eta\,\times$ iterations, so
every row below runs the same amount of diffusion and differs only in how finely it is
discretised.

In [ ]:
T_FIXED = CFG["eta"] * CFG["n_steps"]
ETAS = [CFG["eta"], CFG["eta"] / 2, CFG["eta"] / 4]
REFINE = {}
for key in DS:
    tgt, ref = REF[key]["target"], REF[key]["ref"]
    rows = []
    for eta in ETAS:
        n_steps = int(round(T_FIXED / eta))
        row = {"eta": "{:.1e}".format(eta), "iterations": n_steps}
        for kind, s in [("none", 0.0), ("const", S_MAIN), ("axial", S_MAIN)]:
            r = E.chain(key, kind, s, eta=eta, n_steps=n_steps, ref=None, track=False)
            sc = E.score(ref, r["x"], tgt.R)
            lab = E.METHOD_LABEL[kind].replace("$", "")
            row[lab + " : W1"] = round(sc["W1_mean"], 4)
            row[lab + " : dK"] = "{:.2%}".format(sc["boundary"])
            row[lab + " : atom/sqrt(eta)"] = round(sc["boundary"] / np.sqrt(eta), 2)
        rows.append(row)
    REFINE[key] = pd.DataFrame(rows).set_index("eta")
    print("\n== {} ==  fixed physical time T = {:.2f}".format(DS[key].title, T_FIXED))
    display(REFINE[key].T)

## Summary